In [1]:
import pandas as pd
import numpy as np

# ==========================
# LOAD DATA
# ==========================

file="../11. Near Miss- 2019.xlsx"

xls=pd.ExcelFile(file)

print("Sheets:")
print(xls.sheet_names)

sheet=xls.sheet_names[0]

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

print("\nOriginal Shape:",df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

empty_values=[

    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan",
    "Not Mentioned",
    "not mentioned",
    "NOT MENTIONED"
]

df=df.replace(
    empty_values,
    np.nan
)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    empty=(
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    )

    if empty:

        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# FIND DESCRIPTION COLUMNS
# ==========================

details_col=None
event_col=None

for col in df.columns:

    c=col.lower()

    if (
        "details" in c
        and
        "near" in c
    ):

        details_col=col

    if (
        "description" in c
        and
        "event" in c
    ):

        event_col=col

# ==========================
# REMOVE ROW IF BOTH EMPTY
# ==========================

if details_col and event_col:

    before=len(df)

    keep=(
        (
            df[details_col]
            .fillna("")
            .str.strip()
            !=""
        )
        |
        (
            df[event_col]
            .fillna("")
            .str.strip()
            !=""
        )
    )

    df=df[keep]

    print(
        "\nRows Removed (Both empty):",
        before-len(df)
    )

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print(
    "\nDuplicates Removed:",
    duplicates
)

# ==========================
# RESET SERIAL NUMBER
# ==========================

existing=[]

for col in df.columns:

    c=(
        str(col)
        .lower()
        .replace("_","")
        .replace(":","")
        .replace(".","")
        .replace(" ","")
    )

    if (
        c in [
            "sino",
            "slno",
            "serialno"
        ]
        or
        "si"==c
    ):

        existing.append(col)

if existing:

    df=df.drop(
        columns=existing,
        errors="ignore"
    )

df=df.reset_index(
    drop=True
)

df.insert(
    0,
    "SI_No",
    np.arange(
        1,
        len(df)+1
    )
)

print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_11_Near_Miss_2019.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Sheets:
['Near Miss- 2019']


C:\Users\vinyt\tfenv\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\vinyt\AppData\Local\Temp\ipykernel_28436\3828838068.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.replace(



Original Shape: (1213, 36)

Removed Empty Columns:
['Corrective/Preventive_Action', 'Pic', 'Corrective/Preventive_Due_Date', 'Status_of_Corrective/Preventive', 'Closure_Date_-_Corrective/Preventive', 'Remarks_-_Corrective/Preventive']

Rows Removed (Both empty): 0

Duplicates Removed: 0

SI_No reordered

Missing Values Summary:
Port_Name                                       814
Possibility_Of_Recurrence                        14
Details_Of_NearMiss                               5
Description_of_event_leading_to_the_incident    200
Immediate_Action_Taken                            8
Potential_Extent_Of_Damage/Injury               241
Cause_Analysis                                   11
Proposed_Corrective/Preventive_Action           211
Details_Of_Potential_loss_Category              142
Immediate_Action_Initiated                      145
Learning_Potential                              266
Severity_Of_Incident                            201
Area_Of_Concern                              